# AI-Driven QSAR for δ-Opioid Receptor Ligands

**Reproducible research implementation based on the published computational workflow of Dr. Zeynab Fakhar et al.**

This notebook presents a clean implementation of the machine-learning component of the published study, focusing on molecular descriptor-based QSAR, mutual-information feature selection, Random Forest, XGBoost, cross-validation, held-out testing, and prediction analysis.

**Reference:** Fakhar Z. et al. *Revealing key structural features for developing new agonists targeting δ opioid receptor: Combined machine learning and molecular modeling perspective.* Medicine in Drug Discovery (2024).

> This public implementation is organized for reproducibility. It should not be described as an exact reproduction of the published numerical results unless the original data, preprocessing, software versions, and settings are reproduced exactly.

## 1. Environment

The original research notebook starts from a descriptor matrix generated before this modeling stage. This repository therefore expects an approved CSV containing molecular descriptors and a `pKi` target column.

Recommended packages: `pandas`, `numpy`, `scikit-learn`, `xgboost`, `matplotlib`, `seaborn`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Environment loaded successfully.")

## 2. Load descriptor data

The uploaded research notebook contains a descriptor table at this stage of the workflow with 1,793 compounds and 520 columns. The public repository should contain only data that you are permitted to redistribute.

Place the approved dataset at `data/all_descs.csv`.

In [ ]:
DATA_PATH = "data/all_descs.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
display(df.head())

## 3. Define target and molecular descriptor matrix

`pKi` is treated as the regression target. Identifier columns such as ChEMBL IDs are excluded from the numerical descriptor matrix.

In [ ]:
TARGET = "pKi"
ID_COLUMNS = [c for c in ["ChEMBL ID", "SMILES", "Compound ID"] if c in df.columns]

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' was not found.")

y = pd.to_numeric(df[TARGET], errors="coerce")
X = df.drop(columns=[TARGET] + ID_COLUMNS).copy()
X = X.apply(pd.to_numeric, errors="coerce")
X = X.dropna(axis=1, how="all")

valid = y.notna()
X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].reset_index(drop=True)

print("X shape:", X.shape)
print("y shape:", y.shape)

## 4. Train/test split

The held-out test set is created **before supervised feature selection**. This prevents information from the test set from influencing selection of predictive descriptors.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

## 5. Descriptor cleaning

Missing descriptor values are imputed using training-set medians. Constant descriptors are then removed using training-set variance.

In [ ]:
train_medians = X_train.median(numeric_only=True)

X_train_imp = X_train.fillna(train_medians)
X_test_imp = X_test.fillna(train_medians)

variance = X_train_imp.var(axis=0)
non_constant = variance > 0

X_train_clean = X_train_imp.loc[:, non_constant]
X_test_clean = X_test_imp.loc[:, non_constant]

print("Descriptors before cleaning:", X_train.shape[1])
print("Descriptors after cleaning:", X_train_clean.shape[1])

## 6. K-best feature selection

The published workflow explored different numbers of selected descriptors using `SelectKBest` and mutual-information regression.

Here selection is fitted **only on the training set**, avoiding test-set leakage.

Change `K_FEATURES` to reproduce a different K-best experiment.

In [ ]:
K_FEATURES = min(20, X_train_clean.shape[1])

selector = SelectKBest(
    score_func=mutual_info_regression,
    k=K_FEATURES
)

X_train_kbest = selector.fit_transform(X_train_clean, y_train)
X_test_kbest = selector.transform(X_test_clean)

selected_features = X_train_clean.columns[selector.get_support()]

print(f"Selected features: {K_FEATURES}")
print(list(selected_features))

## 7. Inspect selected descriptor scores

In [ ]:
scores = pd.Series(
    selector.scores_, index=X_train_clean.columns
).sort_values(ascending=False)

display(scores.head(K_FEATURES).to_frame("Mutual information score"))

plt.figure(figsize=(9, 5))
scores.head(K_FEATURES).sort_values().plot(kind="barh")
plt.xlabel("Mutual information score")
plt.ylabel("Descriptor")
plt.title("Top selected molecular descriptors")
plt.tight_layout()
plt.show()

## 8. Random Forest regression

Random Forest is one of the ensemble-learning approaches represented in the research workflow.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=500,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train_kbest, y_train)

rf_train_pred = rf.predict(X_train_kbest)
rf_test_pred = rf.predict(X_test_kbest)

rf_metrics = {
    "Train R2": r2_score(y_train, rf_train_pred),
    "Test R2": r2_score(y_test, rf_test_pred),
    "Test RMSE": mean_squared_error(y_test, rf_test_pred) ** 0.5,
    "Test MAE": mean_absolute_error(y_test, rf_test_pred)
}

pd.Series(rf_metrics)

## 9. XGBoost regression

The uploaded research notebook uses XGBoost with a configuration including 1,000 estimators, maximum depth 7, learning rate 0.1, subsampling 0.7 and column subsampling 0.8. These settings are retained here as a starting point for the public implementation.

In [ ]:
xgb = XGBRegressor(
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.10,
    subsample=0.70,
    colsample_bytree=0.80,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb.fit(X_train_kbest, y_train)

xgb_train_pred = xgb.predict(X_train_kbest)
xgb_test_pred = xgb.predict(X_test_kbest)

xgb_metrics = {
    "Train R2": r2_score(y_train, xgb_train_pred),
    "Test R2": r2_score(y_test, xgb_test_pred),
    "Test RMSE": mean_squared_error(y_test, xgb_test_pred) ** 0.5,
    "Test MAE": mean_absolute_error(y_test, xgb_test_pred)
}

pd.Series(xgb_metrics)

## 10. Compare model performance

In [ ]:
comparison = pd.DataFrame(
    [rf_metrics, xgb_metrics],
    index=["Random Forest", "XGBoost"]
)
display(comparison)

## 11. Five-fold cross-validation

Cross-validation is performed on the training data. The held-out test set remains untouched until final evaluation.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_cv_r2 = cross_val_score(
    rf, X_train_kbest, y_train, cv=cv, scoring="r2", n_jobs=-1
)

xgb_cv_r2 = cross_val_score(
    xgb, X_train_kbest, y_train, cv=cv, scoring="r2", n_jobs=-1
)

cv_results = pd.DataFrame({
    "Random Forest": rf_cv_r2,
    "XGBoost": xgb_cv_r2
})

display(cv_results)

print("Mean RF CV R2:", rf_cv_r2.mean())
print("Mean XGBoost CV R2:", xgb_cv_r2.mean())

## 12. Experimental versus predicted pKi

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, xgb_test_pred, alpha=0.70)

lims = [
    min(y_test.min(), xgb_test_pred.min()),
    max(y_test.max(), xgb_test_pred.max())
]
plt.plot(lims, lims, linestyle="--")

plt.xlabel("Experimental pKi")
plt.ylabel("Predicted pKi")
plt.title("XGBoost: Experimental vs Predicted pKi")
plt.tight_layout()
plt.show()

## 13. Residual analysis

In [ ]:
residuals = y_test - xgb_test_pred

plt.figure(figsize=(8, 5))
plt.scatter(xgb_test_pred, residuals, alpha=0.70)
plt.axhline(0, linestyle="--")

plt.xlabel("Predicted pKi")
plt.ylabel("Residual (Experimental - Predicted)")
plt.title("XGBoost residual analysis")
plt.tight_layout()
plt.show()

## 14. Export test predictions

In [ ]:
predictions = pd.DataFrame({
    "Experimental_pKi": y_test.to_numpy(),
    "Predicted_pKi": xgb_test_pred,
    "Residual": residuals.to_numpy()
})

predictions.to_csv("xgboost_test_predictions.csv", index=False)
display(predictions.head())

## 15. Reproducibility and future extensions

This implementation demonstrates:

**molecular descriptors → training/test split → training-only feature selection → ensemble ML → cross-validation → held-out evaluation → prediction analysis**
